# Advanced Mendeley Web Scraper - Google Colab Edition

This notebook provides a complete, ready-to-run web scraping solution for Mendeley with:
- Ethical scraping practices (rate limiting, user-agent rotation)
- Error handling and retry mechanisms
- Anti-blocking measures
- Comprehensive logging and reporting

**Important:** This scraper respects website policies and implements delays to avoid overloading servers.

### the categories We Will work with

In [ ]:
categories = {
    # Mohammed Alyafrosy
    'Computer_Science': [
        'artificial intelligence', 'cloud computing', 'computation theory',
        'computer architecture', 'computer graphics', 'cyber security',
        'database systems', 'human interaction', 'programming languages',
        'quantum computing', 'software engineering'
    ],
    'Medicine': [
        'orthopedics', 'ophthalmology', 'cardiology', 'surgery',
        'neurology', 'oncology', 'public health', 'radiology',
        'pediatrics', 'immunology', 'dermatology', 'psychiatry',
    ],
    'Business': [
        'business law', 'marketing', 'finance', 'management',
        'entrepreneurship', 'accounting', 'human resources', 'supply chain',
        'business intelligence', 'e-commerce', 'project management', 'corporate governance',
    ],

    # Ryadh Alizi
    'Biology': [
        'genetics','immunology', 'ecology','zoology',
        'botany', 'evolutionary biology', 'marine biology',
        'cell biology', 'biochemistry', 'physiology',
    ],
    'Physics': [
        'quantum mechanics', 'thermodynamics','astrophysics',
        'particle physics', 'condensed matter','nuclear physics',
        'fluid dynamics', 'cosmology','optics', 'string theory'
    ],
    # #============================UnExeptable================================
    # 'Engineering': [
    #     'aerospace engineering', 'biomedical engineering', 'chemical engineering',
    #     'civil engineering', 'computer engineering', 'electrical engineering',
    #     'environmental engineering', 'industrial engineering',
    #     'materials science', 'mechanical engineering'
    # ],
    # #======================================================================

    # Emran Nasser
    'Chemistry': [
        'inorganic chemistry','Electro chemistry', 'green chemistry',
        'organic chemistry', 'analytical chemistry', 'environmental chemistry',
        'biochemistry', 'polymer chemistry', 'medicinal chemistry',
        'food chemistry', 'materials chemistry', 'theoretical chemistry',
    ],
    'Mathematics': [
        'algebra', 'calculus', 'statistics', 'probability',
        'number theory', 'topology', 'differential equations',
        'mathematical modeling','optimization', 'combinatorics',
    ],
    'Psychology': [
        'clinical psychology', 'cognitive psychology', 'developmental psychology',
        'social psychology', 'forensic psychology', 'health psychology',
        'educational psychology', 'neuropsychology', 'industrial organizational psychology',
        'sports psychology','comparative psychology', 'positive psychology',
    ],
    'Environmental Science': [
        'climate change', 'conservation', 'ecology', 'pollution',
        'sustainability', 'environmental policy','oceanography', 'geology',
        'meteorology', 'renewable energy','environmental economics',
    ],

}


## Step 1: Install Required Dependencies

In [ ]:
# Install all required packages
!pip install -q selenium pandas beautifulsoup4 requests

# Install Chrome and ChromeDriver for Colab
!apt-get update
!apt-get install -y chromium-chromedriver

# Set up ChromeDriver path for Colab
import sys
sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

print("✓ All dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 30.0 MB/s eta 0:00:00
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illi

## Step 2: Import Libraries and Verify Setup

In [ ]:
# Verify all imports work correctly
import requests
import random
import time
import pandas as pd
from bs4 import BeautifulSoup
import logging
import json
import os
from datetime import datetime
import re
from queue import Queue
import threading

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service

print("✓ All libraries imported successfully!")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Selenium version: {webdriver.__version__}")

✓ All libraries imported successfully!
✓ Pandas version: 2.2.2
✓ Selenium version: 4.38.0


## Step 3: Define the Scraper Class (Modified for Abstract Extraction)

In [ ]:
class AdvancedMendeleyScraper:
    def __init__(self, headless=True):
        """Initialize the scraper with ethical defaults"""
        # User agents for rotation
        self.user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.6261.94 Safari/537.36',
            'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.6099.110 Safari/537.36',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0',
            'Mozilla/5.0 (Windows NT 6.3; Win64; x64; rv:108.0) Gecko/20100101 Firefox/108.0',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_12_6) AppleWebKit/603.3.8 (KHTML, like Gecko) Version/10.1.2 Safari/603.3.8',
            'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.6045.159 Safari/537.36',
            'Mozilla/5.0 (X11; Fedora; Linux x86_64; rv:115.0) Gecko/20100101 Firefox/115.0',
            'Mozilla/5.0 (Linux; Android 13; SM-G998B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.5615.48 Mobile Safari/537.36',
            'Mozilla/5.0 (Linux; Android 11; M2101K7AG) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.5414.118 Mobile Safari/537.36',
            'Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.6 Mobile/15E148 Safari/604.1',
            'Mozilla/5.0 (iPad; CPU OS 15_7 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.7 Mobile/15E148 Safari/604.1',
            'Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.5993.94 Safari/537.36',
            'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.5938.132 Safari/537.36',
        ]

        self.setup_logging()
        self.setup_selenium(headless)

        # Define categories to scrape
        self.categories = categories

        self.session_stats = {
            'total_articles': 0,
            'start_time': datetime.now()
        }

    def setup_logging(self):
        """Setup logging system"""
        self.logger = logging.getLogger('MendeleyScraper')
        self.logger.setLevel(logging.INFO)

        if self.logger.handlers:
            self.logger.handlers.clear()

        formatter = logging.Formatter(
            '%(asctime)s - %(levelname)s - %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )

        # File handler
        try:
            file_handler = logging.FileHandler(
                f'mendeley_scraping_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log',
                encoding='utf-8'
            )
            file_handler.setFormatter(formatter)
            self.logger.addHandler(file_handler)
        except Exception as e:
            print(f"[Warning] Failed to create log file: {e}")

        # Console handler
        console_handler = logging.StreamHandler()
        console_handler.setFormatter(formatter)
        self.logger.addHandler(console_handler)

    def get_random_user_agent(self):
        """Get a random user agent"""
        return random.choice(self.user_agents)

    def setup_selenium(self, headless=True):
        """Setup Selenium WebDriver for Colab environment"""
        chrome_options = Options()

        # Essential for Colab
        if headless:
            chrome_options.add_argument("--headless=new")

        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation", "enable-logging"])
        chrome_options.add_experimental_option('useAutomationExtension', False)

        # Performance optimization
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--disable-software-rasterizer")
        chrome_options.add_argument("--disable-extensions")
        chrome_options.add_argument("--disable-infobars")

        # Random User-Agent
        chrome_options.add_argument(f"--user-agent={self.get_random_user_agent()}")

        try:
            self.driver = webdriver.Chrome(options=chrome_options)

            # Hide automation property
            self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            self.driver.set_page_load_timeout(30)
            self.driver.set_script_timeout(30)
            self.wait = WebDriverWait(self.driver, 20)
            self.logger.info("[Done] WebDriver successfully initialized")

        except Exception as e:
            self.logger.error(f"[Failed] Failed to initialize WebDriver: {e}")
            raise

    def random_delay(self, min_delay=0.6, max_delay=0.2):
        """Random delay to simulate human behavior"""
        delay = random.uniform(min_delay, max_delay)
        time.sleep(delay)

    def smart_delay(self, base_delay=0.3, variation=0.2):
        """Smart delay with random variation"""
        delay = base_delay + random.uniform(-variation, variation)
        delay = max(1, delay)
        time.sleep(delay)

    def rotate_user_agent(self):
        """Rotate User-Agent to avoid detection"""
        new_ua = self.get_random_user_agent()
        try:
            self.driver.execute_cdp_cmd('Network.setUserAgentOverride', {"userAgent": new_ua})
            self.logger.debug("[Change] User-Agent rotated")
        except Exception as e:
            self.logger.debug(f"[Warning] Could not rotate user agent: {e}")

    def safe_get_page(self, url, max_retries=3):
        """Safely load a page with retries"""
        for attempt in range(max_retries):
            try:
                self.driver.get(url)
                return True
            except Exception as e:
                self.logger.warning(f"[Warning] Attempt {attempt + 1} failed to load page: {e}")
                if attempt < max_retries - 1:
                    time.sleep(0.3)
        return False

    def wait_for_page_load(self):
        """Wait for page to fully load"""
        try:
            # Wait for body
            self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

            # Extended wait for JavaScript to render articles
            time.sleep(0.5)

            # Scroll to trigger lazy loading
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(0.3)
            self.driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(0.4)

            return True
        except TimeoutException:
            self.logger.warning("[Timeout] Page load timeout")
            return False

    def is_no_results_page(self):
        """Check if the page has no results"""
        try:
            page_source = self.driver.page_source.lower()
            no_results_indicators = [
                'no results found',
                'no articles found',
                'no matching results',
                'your search did not match'
            ]
            return any(indicator in page_source for indicator in no_results_indicators)
        except:
            return False

    def get_main_category(self, tag):
        """Determine the main category from the sub-tag"""
        for main_cat, sub_tags in self.categories.items():
            if tag in sub_tags:
                return main_cat
        return "Other"

    def save_progress(self, tag, articles):
        """Save the current progress to a CSV file"""
        try:
            df = pd.DataFrame(articles)
            file_name = f"mendeley_data_{tag.replace(' ', '_')}.csv"
            df.to_csv(file_name, index=False, encoding='utf-8')
            self.logger.info(f"[Save] Progress saved to {file_name} with {len(articles)} articles")
        except Exception as e:
            self.logger.error(f"[Failed] Could not save progress: {e}")

    def search_articles_by_tag(self, tag, max_pages=2):
        """Search for articles by tag with pagination"""
        all_articles = []
        consecutive_empty_pages = 0

        self.logger.info(f"[Search] Starting search for: '{tag}' ({max_pages} pages)")

        for page in range(1, max_pages + 1):
            self.logger.info(f"[Page] Processing page {page}/{max_pages} for: '{tag}'")

            try:
                # Build search URL
                search_url = f"https://www.mendeley.com/search/?page={page}&query={tag.replace(' ', '%20')}"

                # Load page
                if not self.safe_get_page(search_url):
                    self.logger.warning(f"[Skip] Skipping page {page} due to loading issue")
                    continue

                # Wait for load
                if not self.wait_for_page_load():
                    self.logger.warning(f"[Skip] Skipping page {page} due to timeout")
                    continue

                # Check for no results
                if self.is_no_results_page():
                    consecutive_empty_pages += 1
                    self.logger.warning(f"[No Results] No results found on page {page}")
                    if consecutive_empty_pages >= 2:
                        self.logger.info("[Stop] Stopping search due to consecutive empty pages")
                        break
                    continue
                else:
                    consecutive_empty_pages = 0

                # Extract articles using improved function
                page_articles = self.extract_articles_from_current_page_improved(tag, page)

                if not page_articles:
                    self.logger.warning(f"[Warning] No articles extracted from page {page}")
                    continue

                all_articles.extend(page_articles)
                self.logger.info(f"[Done] Extracted {len(page_articles)} articles from page {page}")

                # Save progress
                self.save_progress(tag, all_articles)

                # Ethical delay between pages
                self.smart_delay(4, 2)

                # Rotate User-Agent every 5 pages
                if page % 5 == 0:
                    self.rotate_user_agent()

            except Exception as e:
                self.logger.error(f"[Failed] Error on page {page}: {e}")
                continue

        self.logger.info(f"[Completed] Search for '{tag}' completed - {len(all_articles)} articles")
        return all_articles

    def extract_articles_from_current_page_improved(self, tag, page_number):
        """Improved article extraction without threading to avoid connection issues"""
        articles = []

        try:
            # Improved selectors for article elements
            article_selectors = [
                "[data-testid='search-result-item']",
                ".SearchResults__item",
                ".SearchResultItem",
                "article[class*='result']",
                "div[class*='result']",
                "li[class*='result']"
            ]

            article_elements = []
            for selector in article_selectors:
                try:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    if elements:
                        article_elements = elements
                        self.logger.info(f"[Found] Using selector: {selector}")
                        break
                except:
                    continue

            if not article_elements:
                self.logger.warning("[Warning] No article elements found with any selector")
                return articles

            self.logger.info(f"[Search] Found {len(article_elements)} potential articles on page {page_number}")

            # Process articles sequentially to avoid connection issues
            successful_articles = 0
            for idx, article_elem in enumerate(article_elements, 1):
                try:
                    self.logger.info(f"[Processing] Article {idx}/{len(article_elements)}")

                    # Extract basic data first
                    article_data = self.extract_article_basic_info_improved(article_elem, tag, page_number, idx)

                    if article_data and article_data.get('title') and len(article_data['title']) > 5:
                        # Extract abstract with proper waiting
                        full_abstract = self.extract_abstract_safely(article_elem, article_data)
                        article_data['full_abstract'] = full_abstract

                        articles.append(article_data)
                        successful_articles += 1
                        self.logger.info(f"[Success] Processed article {idx}: {article_data['title'][:50]}...")
                    else:
                        self.logger.warning(f"[Skip] Article {idx} has insufficient data")

                except Exception as e:
                    self.logger.error(f"[Failed] Error processing article {idx}: {e}")
                    continue

                # Small delay between articles
                # time.sleep(1)

            self.logger.info(f"[Completed] Successfully processed {successful_articles}/{len(article_elements)} articles")
            return articles

        except Exception as e:
            self.logger.error(f"[Failed] Error extracting articles from page: {e}")
            return []

    def extract_article_basic_info_improved(self, article_elem, tag, page_num, article_idx):
        """Improved basic information extraction with better selectors"""
        article_data = {
            'tag': tag,
            'main_category': self.get_main_category(tag),
            'page': page_num,
            'position': article_idx,
            'scraped_at': datetime.now().isoformat(),
            'title': '',
            'url': '',
            'authors': '',
            'year': '',
            'abstract_snippet': '',
            'publisher': '',
            'citations': '0',
            'readers': '0',
            'doi': '',
            'open_access': 'No'
        }

        try:
            # Improved title extraction - أكثر مرونة
            title_selectors = [
                "[data-testid='title']",
                ".qe-title",
                "h3",
                "h2",
                "h4",
                "[class*='title']",
                "a[class*='title']",
                ".title",
                "span[class*='title']"
            ]

            for selector in title_selectors:
                try:
                    title_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    title_text = title_elem.text.strip()
                    if title_text and len(title_text) > 3:
                        article_data['title'] = title_text

                        # Try to get URL from title element
                        try:
                            href = title_elem.get_attribute('href')
                            if href and 'http' in href:
                                article_data['url'] = href
                        except:
                            pass
                        break
                except:
                    continue

            # Improved authors extraction
            author_selectors = [
                "[data-testid='authors']",
                ".qe-authors",
                "[class*='author']",
                "[class*='creator']",
                ".authors",
                "span[class*='author']",
                "div[class*='author']"
            ]

            for selector in author_selectors:
                try:
                    authors_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    author_text = authors_elem.text.strip()
                    if author_text and len(author_text) > 2:
                        article_data['authors'] = author_text
                        break
                except:
                    continue

            # Improved year extraction
            year_selectors = [
                "[data-testid='year']",
                ".qe-year",
                "[class*='year']",
                "time",
                ".year",
                "span[class*='year']"
            ]

            for selector in year_selectors:
                try:
                    year_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    year_text = year_elem.text.strip()
                    # Extract year using regex for better accuracy
                    year_match = re.search(r'\b(19|20)\d{2}\b', year_text)
                    if year_match:
                        article_data['year'] = year_match.group()
                        break
                    elif year_text and year_text.isdigit() and len(year_text) == 4:
                        article_data['year'] = year_text
                        break
                except:
                    continue

            # Abstract snippet - أكثر مرونة
            abstract_selectors = [
                "[data-testid='abstract']",
                ".qe-abstract-snippet",
                "[class*='abstract']",
                "[class*='snippet']",
                ".abstract",
                "div[class*='description']",
                "p[class*='abstract']"
            ]
            print("============================================================")
            for selector in abstract_selectors:
                try:
                    abstract_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    abstract_text = abstract_elem.text.strip()
                    if abstract_text and len(abstract_text) > 10:
                        article_data['abstract_snippet'] = abstract_text
                        break
                except:
                    continue

            # Additional metadata
            self.extract_additional_info_improved(article_elem, article_data)

        except Exception as e:
            self.logger.warning(f"[Warning] Error extracting basic info: {e}")

        return article_data

    def extract_additional_info_improved(self, article_elem, article_data):
        """Improved additional metadata extraction"""
        try:
            # Publisher
            publisher_selectors = [
                "[data-testid='publisher']",
                ".qe-publication",
                "[class*='publisher']",
                "[class*='journal']",
                ".publisher",
                "span[class*='publisher']"
            ]

            for selector in publisher_selectors:
                try:
                    publisher_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    publisher_text = publisher_elem.text.strip()
                    if publisher_text:
                        article_data['publisher'] = publisher_text
                        break
                except:
                    continue

            # Citations
            citation_selectors = [
                "[data-testid='citations']",
                ".qe-citations-count",
                "[class*='citation']",
                ".citations",
                "span[class*='citation']"
            ]

            for selector in citation_selectors:
                try:
                    citations_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    citation_text = citations_elem.text.strip()
                    if citation_text:
                        # Extract numbers only
                        numbers = re.findall(r'\d+', citation_text)
                        if numbers:
                            article_data['citations'] = numbers[0]
                        break
                except:
                    continue

            # Readers
            reader_selectors = [
                "[data-testid='readers']",
                ".qe-readers-count",
                "[class*='reader']",
                ".readers",
                "span[class*='reader']"
            ]

            for selector in reader_selectors:
                try:
                    readers_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    reader_text = readers_elem.text.strip()
                    if reader_text:
                        # Extract numbers only
                        numbers = re.findall(r'\d+', reader_text)
                        if numbers:
                            article_data['readers'] = numbers[0]
                        break
                except:
                    continue

            # DOI
            doi_selectors = [
                "[data-testid='doi']",
                ".qe-article-card-doi",
                "[class*='doi']",
                ".doi",
                "span[class*='doi']"
            ]

            for selector in doi_selectors:
                try:
                    doi_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    doi_text = doi_elem.text.strip()
                    if doi_text and '10.' in doi_text:
                        article_data['doi'] = doi_text
                        break
                except:
                    continue

            # Open Access
            oa_selectors = [
                "[data-testid='open-access']",
                ".qe-open-access",
                "[class*='open']",
                "[class*='access']",
                ".open-access",
                "[class*='oa']"
            ]

            for selector in oa_selectors:
                try:
                    oa_elem = article_elem.find_element(By.CSS_SELECTOR, selector)
                    if oa_elem.is_displayed():
                        article_data['open_access'] = "Yes"
                        break
                except:
                    continue

        except Exception as e:
            self.logger.warning(f"[Warning] Error extracting additional info: {e}")

    def extract_abstract_safely(self, article_elem, article_data):
        """Safely extract abstract with proper waiting for page updates"""
        try:
            return self.click_for_full_abstract_improved(article_elem, article_data)
        except Exception as e:
            self.logger.warning(f"[Warning] Failed to extract abstract: {e}")
            return article_data.get('abstract_snippet', '')

    def click_for_full_abstract_improved(self, article_elem, article_data):
        """Improved click method with proper waiting for page updates"""
        try:
            # Save current URL and state
            original_url = self.driver.current_url
            original_title = self.driver.title

            # Find clickable element
            click_selectors = [
                "[data-testid='title']",
                ".qe-title",
                "h3 a",
                "a[class*='title']",
                "h3",
                "h2",
                ".title a"
            ]

            clickable_element = None
            for selector in click_selectors:
                try:
                    clickable_element = article_elem.find_element(By.CSS_SELECTOR, selector)
                    if clickable_element.is_displayed() and clickable_element.is_enabled():
                        break
                    else:
                        clickable_element = None
                except:
                    continue

            if not clickable_element:
                self.logger.warning("[Warning] No clickable element found")
                return ""

            # Scroll to element and click using JavaScript
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", clickable_element)
            time.sleep(0.2)

            # Store current article count for verification
            initial_article_count = len(self.driver.find_elements(By.CSS_SELECTOR, "li[class*='result']"))

            # Click using JavaScript to avoid interception
            self.driver.execute_script("arguments[0].click();", clickable_element)

            # Wait for page to update - increased waiting time
            time.sleep(0.2)

            # Check if we're still on the same page (modal might have opened)
            current_url = self.driver.current_url
            if current_url == original_url:
                # Probably a modal opened, look for abstract in modal
                self.logger.info("[Info] Modal detected, searching for abstract in modal")
                abstract_text = self.extract_abstract_from_modal()
            else:
                # New page loaded, extract abstract
                self.logger.info("[Info] New page detected, extracting abstract")
                abstract_text = self.extract_abstract_from_detail_page()

                # Navigate back to search results
                self.driver.back()
                time.sleep(0.4)

                # Wait for search results to reload
                self.wait_for_page_load()

            return abstract_text

        except Exception as e:
            self.logger.error(f"[Error] Abstract extraction failed: {e}")
            try:
                # Try to navigate back if we're stuck
                if self.driver.current_url != original_url:
                    self.driver.back()
                    time.sleep(0.6)
                    self.wait_for_page_load()
            except:
                pass
            return ""

    def extract_abstract_from_modal(self):
        """Extract abstract from modal dialog"""
        try:
            # Wait for modal to appear
            time.sleep(0.6)

            # Look for abstract in modal
            abstract_selectors = [
                "[data-testid='abstract']",
                ".Abstract__abstract",
                "[class*='abstract']",
                ".qe-abstract",
                "div[class*='description']",
                ".abstract-content",
                ".abstract-text"
            ]

            for selector in abstract_selectors:
                try:
                    abstract_elem = self.driver.find_element(By.CSS_SELECTOR, selector)
                    abstract_text = abstract_elem.text.strip()
                    if abstract_text and len(abstract_text) > 50:
                        # Try to close modal
                        self.close_modal()
                        return abstract_text
                except:
                    continue

            # If no abstract found, close modal and return empty
            self.close_modal()
            return ""

        except Exception as e:
            self.logger.warning(f"[Warning] Modal abstract extraction failed: {e}")
            return ""

    def extract_abstract_from_detail_page(self):
        """Extract abstract from detail page"""
        try:
            # Wait for page to load completely
            time.sleep(0.6)

            # Look for abstract on detail page
            abstract_selectors = [
                "[data-testid='abstract']",
                ".Abstract__abstract",
                "[class*='abstract']",
                ".qe-abstract",
                "div[class*='description']",
                ".abstract-content",
                ".abstract-text",
                "section[class*='abstract']",
                "div.abstract",
                "p.abstract"
            ]

            for selector in abstract_selectors:
                try:
                    abstract_elems = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    for elem in abstract_elems:
                        text = elem.text.strip()
                        if text and len(text) > 50:
                            return text
                except:
                    continue

            return ""

        except Exception as e:
            self.logger.warning(f"[Warning] Detail page abstract extraction failed: {e}")
            return ""

    def close_modal(self):
        """Close modal dialog if present"""
        try:
            # Try various close button selectors
            close_selectors = [
                "[data-testid='close-button']",
                ".close-button",
                "[class*='close']",
                "button[aria-label*='close']",
                "button[class*='close']",
                ".modal-close",
                "button[data-dismiss='modal']"
            ]

            for selector in close_selectors:
                try:
                    close_btn = self.driver.find_element(By.CSS_SELECTOR, selector)
                    if close_btn.is_displayed():
                        self.driver.execute_script("arguments[0].click();", close_btn)
                        time.sleep(0.5)
                        break
                except:
                    continue
        except Exception as e:
            self.logger.debug(f"[Debug] Could not close modal: {e}")

    def scrape_all(self, max_pages=2):
        """
        Scrape all tags taken from self.tags (if set) or from self.categories values.
        Each tag's results are saved immediately to Google Drive after it finishes.
        """
        import os
        from datetime import datetime

        all_results = []
        results_by_tag = {}

        # Build tags list: prefer self.tags if user set it, otherwise flatten self.categories
        if getattr(self, "tags", None):
            tag_list = list(self.tags)
        else:
            tag_list = []
            for sublist in getattr(self, "categories", {}).values():
                if isinstance(sublist, (list, tuple)):
                    tag_list.extend(sublist)

        # Guard: if still empty, log and return empty DataFrame
        if not tag_list:
            self.logger.warning("[Abort] No tags found in self.tags or self.categories.")
            return pd.DataFrame()

        drive_folder = "./Mendeley_Research/"
        os.makedirs(drive_folder, exist_ok=True)
        self.logger.info(f"[Info] Saving each tag’s results directly to: {drive_folder}")

        for tag in tag_list:
            self.logger.info(f"\n========== [Scraping Tag: {tag}] ==========")
            try:
                # Reuse existing search logic that builds URLs and handles pagination
                tag_results = self.search_articles_by_tag(tag, max_pages=max_pages)
            except Exception as e:
                self.logger.error(f"[Failed] search_articles_by_tag for '{tag}' raised: {e}")
                tag_results = []

            # Save per-tag file to Drive (with timestamp)
            if tag_results:
                df_tag = pd.DataFrame(tag_results)
                safe_tag = re.sub(r'[\\/*?:"<>|]', "_", tag.replace(" ", "_"))
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                file_path = os.path.join(drive_folder, f"mendeley_data_{safe_tag}_{timestamp}.csv")
                try:
                    df_tag.to_csv(file_path, index=False, encoding='utf-8')
                    self.logger.info(f"[Saved] {len(tag_results)} articles saved for tag '{tag}' → {file_path}")
                except Exception as e:
                    self.logger.error(f"[Failed] Could not save CSV for tag '{tag}': {e}")
            else:
                self.logger.warning(f"[Skipped] No results found for tag '{tag}'")

            results_by_tag[tag] = tag_results
            all_results.extend(tag_results)

        total_articles = sum(len(v) for v in results_by_tag.values())
        self.logger.info(f"[Completed] Total articles scraped: {total_articles} across {len(results_by_tag)} tags")
        return pd.DataFrame(all_results)


## Step 4: Run the Scraper

In [ ]:
# ============================================
# 1. Connect Google Drive
# ============================================
from google.colab import drive

# ============================================
# 2. Prepare Drive Folder
# ============================================
import os
drive_folder = './Mendeley_Research/'
os.makedirs(drive_folder, exist_ok=True)
print(f" Drive folder ready: {drive_folder}")

# ============================================
# 3. Initialize the Scraper
# ============================================
scraper = AdvancedMendeleyScraper(headless=True)

# ============================================
# 4. Start Scraping
# ============================================
print("\n Starting scraping process...\n")
results_df = scraper.scrape_all(max_pages=200)

# ============================================
# 5. Show Sample of Results
# ============================================
if not results_df.empty:
    print("\n--- Sample Results ---")
    display(results_df[['title', 'full_abstract', 'authors', 'year']].head(10))
else:
    print(" No results were extracted.")

# ============================================
# 6. Optional: Create a Global Backup
# ============================================
from datetime import datetime
if not results_df.empty:
    backup_file = f"{drive_folder}/mendeley_all_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    results_df.to_csv(backup_file, index=False, encoding='utf-8')
    print(f" Global backup saved at: {backup_file}")

# ============================================
# 7. Clean Up
# ============================================
del scraper
print("\n Scraping complete. Each tag file saved individually in:")
print(f" {drive_folder}")


# Step 5: Merge files


# Research Paper Data Processing Pipeline

## Overview
Data processing and cleaning pipeline for multi-disciplinary research papers from Mendeley dataset. Processes 97 subcategories across 9 major scientific domains.

## Features
- **Multi-domain Integration**: Merges CSV files from Computer Science, Medicine, Business, Chemistry, Mathematics, Psychology, Environmental Science, Biology, and Physics
- **Data Quality Analysis**: Provides comprehensive statistics per category including clean records, uniqueness ratios, and completeness metrics
- **Automated Cleaning**: Removes empty abstracts and duplicates while preserving data integrity
- **Quality Reporting**: Generates detailed quality assessment with export-ready cleaned dataset

## Output
- Cleaned dataset with category labels and abstracts
- Comprehensive data quality report
- Category-wise statistics and global metrics

In [2]:
import os
import pandas as pd
import numpy as np
from google.colab import drive
# ============================================
# 1. Connect Google Drive
# ============================================

# File paths list (each path represents a CSV file within specific category)
categories = [

     './Mendeley_Research/Computer_Science/artificial_intelligence.csv',
     './Mendeley_Research/Computer_Science/cyber_security.csv',
     './Mendeley_Research/Computer_Science/software_engineering.csv',
     './Mendeley_Research/Computer_Science/computer_architecture.csv',
     './Mendeley_Research/Computer_Science/database_systems.csv',
     './Mendeley_Research/Computer_Science/computation_theory.csv',
     './Mendeley_Research/Computer_Science/human_interaction.csv',
     './Mendeley_Research/Computer_Science/computer_graphics.csv',
     './Mendeley_Research/Computer_Science/cloud_computing.csv',
     './Mendeley_Research/Computer_Science/quantum_computing.csv',
     './Mendeley_Research/Computer_Science/programming_languages.csv',
     './Mendeley_Research/Medicine/immunology.csv',
     './Mendeley_Research/Medicine/orthopedics.csv',
     './Mendeley_Research/Medicine/ophthalmology.csv',
     './Mendeley_Research/Medicine/cardiology.csv',
     './Mendeley_Research/Medicine/neurology.csv',
     './Mendeley_Research/Medicine/oncology.csv',
     './Mendeley_Research/Medicine/public_health.csv',
     './Mendeley_Research/Medicine/pediatrics.csv',
     './Mendeley_Research/Medicine/dermatology.csv',
     './Mendeley_Research/Medicine/psychiatry.csv',
     './Mendeley_Research/Medicine/radiology.csv',
     './Mendeley_Research/Medicine/surgery.csv',
     './Mendeley_Research/Business/business_law.csv',
     './Mendeley_Research/Business/marketing.csv',
     './Mendeley_Research/Business/finance.csv',
     './Mendeley_Research/Business/management.csv',
     './Mendeley_Research/Business/entrepreneurship.csv',
     './Mendeley_Research/Business/accounting.csv',
     './Mendeley_Research/Business/human_resources.csv',
     './Mendeley_Research/Business/supply_chain.csv',
     './Mendeley_Research/Business/business_intelligence.csv',
     './Mendeley_Research/Business/e-commerce.csv',
     './Mendeley_Research/Business/project_management.csv',
     './Mendeley_Research/Business/corporate_governance.csv',
     './Mendeley_Research/Chemistry/organic_chemistry.csv',
     './Mendeley_Research/Chemistry/inorganic_chemistry.csv',
     './Mendeley_Research/Chemistry/Electro_chemistry.csv',
     './Mendeley_Research/Chemistry/food_chemistry.csv',
     './Mendeley_Research/Chemistry/green_chemistry.csv',
     './Mendeley_Research/Chemistry/analytical_chemistry.csv',
     './Mendeley_Research/Chemistry/biochemistry.csv',
     './Mendeley_Research/Chemistry/environmental_chemistry.csv',
     './Mendeley_Research/Chemistry/polymer_chemistry.csv',
     './Mendeley_Research/Chemistry/medicinal_chemistry.csv',
     './Mendeley_Research/Chemistry/materials_chemistry.csv',
     './Mendeley_Research/Chemistry/theoretical_chemistry.csv',
     './Mendeley_Research/Mathematics/algebra.csv',
     './Mendeley_Research/Mathematics/calculus.csv',
     './Mendeley_Research/Mathematics/statistics.csv',
     './Mendeley_Research/Mathematics/probability.csv',
     './Mendeley_Research/Mathematics/number_theory.csv',
     './Mendeley_Research/Mathematics/topology.csv',
     './Mendeley_Research/Mathematics/differential_equations.csv',
     './Mendeley_Research/Mathematics/mathematical_modeling.csv',
     './Mendeley_Research/Mathematics/optimization.csv',
     './Mendeley_Research/Mathematics/combinatorics.csv',
     './Mendeley_Research/Psychology/clinical_psychology.csv',
     './Mendeley_Research/Psychology/cognitive_psychology.csv',
     './Mendeley_Research/Psychology/developmental_psychology.csv',
     './Mendeley_Research/Psychology/social_psychology.csv',
     './Mendeley_Research/Psychology/neuropsychology.csv',
     './Mendeley_Research/Psychology/forensic_psychology.csv',
     './Mendeley_Research/Psychology/health_psychology.csv',
     './Mendeley_Research/Psychology/educational_psychology.csv',
     './Mendeley_Research/Psychology/industrial_organizational_psychology.csv',
     './Mendeley_Research/Psychology/sports_psychology.csv',
     './Mendeley_Research/Psychology/comparative_psychology.csv',
     './Mendeley_Research/Psychology/positive_psychology.csv',
     './Mendeley_Research/Environmental_Science/climate_change.csv',
     './Mendeley_Research/Environmental_Science/conservation.csv',
     './Mendeley_Research/Environmental_Science/ecology.csv',
     './Mendeley_Research/Environmental_Science/pollution.csv',
     './Mendeley_Research/Environmental_Science/sustainability.csv',
     './Mendeley_Research/Environmental_Science/environmental_policy.csv',
     './Mendeley_Research/Environmental_Science/oceanography.csv',
     './Mendeley_Research/Environmental_Science/geology.csv',
     './Mendeley_Research/Environmental_Science/meteorology.csv',
     './Mendeley_Research/Environmental_Science/renewable_energy.csv',
     './Mendeley_Research/Environmental_Science/environmental_economics.csv',
     './Mendeley_Research/Biology/genetics.csv',
     './Mendeley_Research/Biology/immunology.csv',
     './Mendeley_Research/Biology/ecology.csv',
     './Mendeley_Research/Biology/zoology.csv',
     './Mendeley_Research/Biology/botany.csv',
     './Mendeley_Research/Biology/evolutionary_biology.csv',
     './Mendeley_Research/Biology/marine_biology.csv',
     './Mendeley_Research/Biology/cell_biology.csv',
     './Mendeley_Research/Biology/biochemistry.csv',
     './Mendeley_Research/Biology/physiology.csv',
     './Mendeley_Research/Physics/quantum_mechanics.csv',
     './Mendeley_Research/Physics/thermodynamics.csv',
     './Mendeley_Research/Physics/astrophysics.csv',
     './Mendeley_Research/Physics/particle_physics.csv',
     './Mendeley_Research/Physics/optics.csv',
     './Mendeley_Research/Physics/condensed_matter.csv',
     './Mendeley_Research/Physics/nuclear_physics.csv',
     './Mendeley_Research/Physics/fluid_dynamics.csv',
     './Mendeley_Research/Physics/cosmology.csv',
     './Mendeley_Research/Physics/string_theory.csv'

]

merged_data = []

# Read and merge files
print("[INFO] Reading and merging dataset files...")
for file_path in categories:
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            merged_data.append(df)
            print(f"[SUCCESS] Loaded: {os.path.basename(file_path)} - {len(df)} records")
        except Exception as e:
            print(f"[ERROR] Failed to load {file_path}: {str(e)}")
    else:
        print(f"[WARNING] File not found: {file_path}")

# Merge all dataframes
if merged_data:
    merged_df = pd.concat(merged_data, ignore_index=True)
    print(f"[SUCCESS] Merged dataset contains {len(merged_df)} total records")
else:
    raise Exception("[CRITICAL] No data files were successfully loaded")

# ============================================================================
# DATA PROCESSING AND ANALYSIS
# ============================================================================

# Select required columns
df = merged_df[['main_category', 'full_abstract']].copy()

# Data cleaning
df['main_category'] = df['main_category'].astype(str).str.strip()
df['full_abstract'] = df['full_abstract'].astype(str).str.strip()

# Remove records with empty category
initial_count = len(df)
df = df[df['main_category'] != '']
print(f"[CLEANING] Removed {initial_count - len(df)} records with empty categories")

def calculate_category_statistics(group):
    """Calculate comprehensive statistics for each category"""
    total = len(group)
    if total == 0:
        return pd.Series({
            'Total_Records': 0,
            'Clean_Abstracts': 0,
            'Unique_Abstracts': 0,
            'Clean_Ratio_Percent': 0.0,
            'Duplicate_Ratio_Percent': 0.0
        })

    # Count non-empty abstracts
    clean_count = group[group['full_abstract'].str.len() > 0].shape[0]
    unique_count = group['full_abstract'].nunique()

    # Calculate ratios
    clean_ratio = round((clean_count / total) * 100, 2) if total > 0 else 0.0
    duplicate_ratio = round(((total - unique_count) / total) * 100, 2) if total > 0 else 0.0

    return pd.Series({
        'Total_Records': total,
        'Clean_Abstracts': clean_count,
        'Unique_Abstracts': unique_count,
        'Clean_Ratio_Percent': clean_ratio,
        'Duplicate_Ratio_Percent': duplicate_ratio
    })

# Generate category-wise statistics
print("[ANALYSIS] Calculating category statistics...")
stats_table = df.groupby('main_category').apply(calculate_category_statistics).reset_index()

# Create comprehensive summary
total_records = stats_table['Total_Records'].sum()
total_clean = stats_table['Clean_Abstracts'].sum()
total_unique_abstracts = df['full_abstract'].nunique()

summary_stats = pd.DataFrame({
    'main_category': ['DATASET_SUMMARY'],
    'Total_Records': [total_records],
    'Clean_Abstracts': [total_clean],
    'Unique_Abstracts': [total_unique_abstracts],
    'Clean_Ratio_Percent': [round((total_clean / total_records) * 100, 2)],
    'Duplicate_Ratio_Percent': [round(((total_records - total_unique_abstracts) / total_records) * 100, 2)]
})

# Combine detailed and summary statistics
final_statistics = pd.concat([stats_table, summary_stats], ignore_index=True)

# Display configuration
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', None)

print("\n" + "="*80)
print("DATA QUALITY ANALYSIS REPORT")
print("="*80)
display(final_statistics.round(2))

# ============================================================================
# DATA CLEANING AND EXPORT
# ============================================================================

print("\n[PROCESSING] Applying data cleaning procedures...")

# Remove empty and duplicate records
initial_size = len(df)
clean_df = df[df['full_abstract'].notna() & (df['full_abstract'] != '')]
clean_df = clean_df.drop_duplicates(subset=['full_abstract'], keep='first')
clean_df = clean_df.reset_index(drop=True)

removed_count = initial_size - len(clean_df)

# Export cleaned dataset
clean_output_path = './cleaned_dataset.csv'
clean_df.to_csv(clean_output_path, index=False, encoding='utf-8')

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("CLEANING PROCESS COMPLETED")
print("="*80)
print(f"Original dataset dimensions: {merged_df.shape}")
print(f"Final cleaned dimensions: {clean_df.shape}")
print(f"Records removed: {removed_count} ({round((removed_count/initial_size)*100, 2)}%)")
print(f"Unique categories in final dataset: {clean_df['main_category'].nunique()}")
print(f"Output file: {clean_output_path}")
print("="*80)

# Additional quality metrics
quality_metrics = {
    'Data_Completeness': f"{round((total_clean / total_records) * 100, 2)}%",
    'Data_Uniqueness': f"{round((total_unique_abstracts / total_records) * 100, 2)}%",
    'Category_Coverage': clean_df['main_category'].nunique(),
    'Final_Abstract_Count': len(clean_df)
}

print("\n[QUALITY_METRICS]")
for metric, value in quality_metrics.items():
    print(f"{metric}: {value}")

print("\n[STATUS] Data processing pipeline completed successfully")


Mounted at /content/drive
[INFO] Reading and merging dataset files...
[SUCCESS] Loaded: artificial_intelligence.csv - 1754 records
[SUCCESS] Loaded: cyber_security.csv - 1323 records
[SUCCESS] Loaded: software_engineering.csv - 1202 records
[SUCCESS] Loaded: computer_architecture.csv - 1410 records
[SUCCESS] Loaded: database_systems.csv - 1345 records
[SUCCESS] Loaded: computation_theory.csv - 1545 records
[SUCCESS] Loaded: human_interaction.csv - 1542 records
[SUCCESS] Loaded: computer_graphics.csv - 1477 records
[SUCCESS] Loaded: cloud_computing.csv - 1388 records
[SUCCESS] Loaded: quantum_computing.csv - 1361 records
[SUCCESS] Loaded: programming_languages.csv - 1219 records
[SUCCESS] Loaded: immunology.csv - 1873 records
[SUCCESS] Loaded: orthopedics.csv - 1349 records
[SUCCESS] Loaded: ophthalmology.csv - 1785 records
[SUCCESS] Loaded: cardiology.csv - 1462 records
[SUCCESS] Loaded: neurology.csv - 1809 records
[SUCCESS] Loaded: oncology.csv - 1599 records
[SUCCESS] Loaded: public

/tmp/ipython-input-2565671360.py:184: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats_table = df.groupby('main_category').apply(calculate_category_statistics).reset_index()



DATA QUALITY ANALYSIS REPORT


,main_category,Total_Records,Clean_Abstracts,Unique_Abstracts,Clean_Ratio_Percent,Duplicate_Ratio_Percent
0,Biology,17931.0,17931.0,16043.0,100.0,10.53
1,Business,16674.0,16674.0,16119.0,100.0,3.33
2,Chemistry,19215.0,19215.0,16722.0,100.0,12.97
3,Computer_Science,15566.0,15566.0,15016.0,100.0,3.53
4,Environmental_Science,17614.0,17614.0,16056.0,100.0,8.85
5,Mathematics,16379.0,16379.0,15542.0,100.0,5.11
6,Medicine,18365.0,18365.0,17037.0,100.0,7.23
7,Physics,17280.0,17280.0,16015.0,100.0,7.32
8,Psychology,18731.0,18731.0,16869.0,100.0,9.94
9,DATASET_SUMMARY,157755.0,157755.0,140004.0,100.0,11.25



[PROCESSING] Applying data cleaning procedures...

CLEANING PROCESS COMPLETED
Original dataset dimensions: (157755, 16)
Final cleaned dimensions: (140004, 2)
Records removed: 17751 (11.25%)
Unique categories in final dataset: 9
Output file: /content/drive/MyDrive/research_papers_Classification/cleaned_dataset.csv

[QUALITY_METRICS]
Data_Completeness: 100.0%
Data_Uniqueness: 88.75%
Category_Coverage: 9
Final_Abstract_Count: 140004

[STATUS] Data processing pipeline completed successfully
